In [ ]:
# cell-01 – Importy

import os
import shutil
import random
from pathlib import Path

In [ ]:
# cell-02

BASE        = Path("/home/marcin/kod/flir-test/data-flir-test")
SRC         = BASE / "projekt-52zdjec"
OUTPUT_DIR  = BASE / "dataset-52"
TRAIN_RATIO = 0.8
RANDOM_SEED = 42

In [ ]:
# cell-03

# Zbieramy pary obraz → etykieta
images = sorted((SRC / "images").glob("*.png"))
pairs  = []
for img in images:
    lbl = SRC / "labels" / img.with_suffix(".txt").name
    if lbl.exists():
        pairs.append((img, lbl))
    else:
        print(f"  [!] Brak etykiety: {img.name}")

print(f"Kompletnych par: {len(pairs)}")

# Podział 80/20
random.seed(RANDOM_SEED)
random.shuffle(pairs)
n_train     = int(len(pairs) * TRAIN_RATIO)
train_pairs = pairs[:n_train]
val_pairs   = pairs[n_train:]
print(f"Train: {len(train_pairs)}  |  Val: {len(val_pairs)}")

# Kopiowanie
for split, split_pairs in [("train", train_pairs), ("val", val_pairs)]:
    (OUTPUT_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)
    for img, lbl in split_pairs:
        shutil.copy2(img, OUTPUT_DIR / "images" / split / img.name)
        shutil.copy2(lbl, OUTPUT_DIR / "labels" / split / lbl.name)

# data.yaml
yaml_content = f"""path: {OUTPUT_DIR}
train: images/train
val:   images/val

nc: 1
names: ['hand']
"""
(OUTPUT_DIR / "data.yaml").write_text(yaml_content)

# Podsumowanie
print("\n" + "═"*40)
print(f"Dataset zapisany: {OUTPUT_DIR}")
for split in ["train", "val"]:
    n = len(list((OUTPUT_DIR / "images" / split).glob("*")))
    print(f"  images/{split}/  →  {n} zdjęć")
print("═"*40)

In [ ]:
# cell-04

from ultralytics import YOLO

model = YOLO("yolov8n-seg.pt")

results = model.train(
    data="/home/marcin/kod/flir-test/data-flir-test/dataset-52/data.yaml",
    epochs=100,
    imgsz=640,
    batch=4,
    device=0,
    project="/home/marcin/kod/flir-test/data-flir-test/runs",
    name="flir-hand-52",
    augment=True,
    degrees=15,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    verbose=True
)

In [ ]:
# cell-05

print(f"Najlepszy model: {results.save_dir}/weights/best.pt")
print(f"\nMetryki:")
print(f"  mAP50:     {results.results_dict.get('metrics/mAP50(M)', 'N/A'):.3f}")
print(f"  mAP50-95:  {results.results_dict.get('metrics/mAP50-95(M)', 'N/A'):.3f}")